# DMRC RAG — 01. Setup & Retrieval Validation

Builds a completely fresh ChromaDB from the current repository and validates retrieval end-to-end:

```
clone repo -> install deps -> auth HF -> download models -> delete old ChromaDB
  -> ingest every Clause JSON + BOQ JSON -> embed -> store -> validate -> retrieval tests
```

**The BOQ ingestion pipeline changed** (`metadata_loader.build_boq_chunk_records`, `text_normalization.build_boq_embedding_input`), so any previously-built `chroma_db/` is obsolete. This notebook always deletes it and rebuilds from scratch — it never reuses an existing collection.

No Gemma weights are loaded here, so this notebook runs fine on a **CPU runtime** (Runtime -> Change runtime type -> CPU) and only needs a GPU to speed up embedding. Once every cell below passes, move to **`02_Gemma_Inference_and_Serving.ipynb`** for the LLM + API part, which does need a GPU runtime.

### 1. Clone the repository (idempotent)

In [ ]:
%cd /content
!test -d dmrc_deploy && (echo "dmrc_deploy/ already present -- pulling latest" && cd dmrc_deploy && git pull) \
    || git clone https://github.com/sunvantaconsultancysolutions-design/dmrc_deploy.git
%cd /content/dmrc_deploy

### 2. Install dependencies

One install, matching `requirements.txt` exactly. `bitsandbytes` / `nvidia-nvjitlink-cu13` are only needed for 4-bit Gemma loading in notebook 2, so they're skipped here (mirrors the Dockerfile's own best-effort split).

In [ ]:
!grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" requirements.txt > /tmp/requirements_core.txt
!pip install -q -r /tmp/requirements_core.txt

In [ ]:
print("Restarting the kernel so numpy loads cleanly after the install above...")
print("Colab will auto-reconnect in a few seconds -- then continue running from the next cell.")
import os
os.kill(os.getpid(), 9)

### 3. Sanity-check the environment

In [ ]:
import torch, transformers, sentence_transformers, chromadb

print(f"torch                : {torch.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")
print(f"transformers         : {transformers.__version__}")
print(f"sentence_transformers: {sentence_transformers.__version__}")
print(f"chromadb             : {chromadb.__version__}")

### 4. Hugging Face authentication

`BAAI/bge-m3` and `BAAI/bge-reranker-v2-m3` are public, but authenticating here avoids anonymous rate limits during the bulk download/embedding step below.

In [ ]:
from huggingface_hub import login
login()

### 5. Download the required models

Pulls `BAAI/bge-m3` (embedding, `batch_embed.py`/`text_normalization` pipeline) and `BAAI/bge-reranker-v2-m3` (cross-encoder, used by the retrieval-validation queries below) into the local HF cache once, up front.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Downloading BAAI/bge-m3 ...")
embed_model = SentenceTransformer("BAAI/bge-m3")
print("Downloading BAAI/bge-reranker-v2-m3 ...")
rerank_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-reranker-v2-m3")
rerank_model = AutoModelForSequenceClassification.from_pretrained("BAAI/bge-reranker-v2-m3")
print("Models downloaded and cached.")

# Free this notebook's own references -- batch_embed.py / reranker.py load their
# own module-level copies when the pipeline below runs; no need to hold two.
del embed_model, rerank_tokenizer, rerank_model

### 6. Delete any existing ChromaDB

The BOQ ingestion pipeline changed since any previously-committed `chroma_db/` was built, so it is obsolete and must never be reused. This step is unconditional.

In [ ]:
import shutil

CHROMA_PATH = "./chroma_db"
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print(f"Deleted existing {CHROMA_PATH}")
else:
    print(f"No existing {CHROMA_PATH} -- nothing to delete")

assert not os.path.exists(CHROMA_PATH), "chroma_db/ still present after delete -- aborting."

### 7. Build a brand-new ChromaDB

Runs `main.py` over every file in `data/` -- three Clause JSON files (`DMRC_Chapter1_transcription.json`, `DMRC_Chapter2_transcription.json`, `chapter3.json`) and three BOQ JSON files (`boq_part1/2/3.json`). `main.py._load_records()` detects each file's shape (`is_boq_json()`) and routes it to `build_chunk_records()` or `build_boq_chunk_records()` accordingly, then embeds with BGE-M3 and stores everything in the fresh `chroma_db/`.

In [ ]:
!python main.py --input-dir data/

### 8. Validate the ChromaDB collection

Read-only check (`src/validate_db.py`) that the collection just built is intact and queryable -- BGE-M3, 1024-dim vectors. Must run as a loose script from the **repo root** (not `cd src` first): `validate_db.py` does a bare `from storage import ...`, which only resolves because Python puts the script's own directory on `sys.path` -- but `storage.py`'s `CHROMA_PATH="./chroma_db"` is resolved against the *current working directory*, which needs to stay the repo root to find the collection just built above.

In [ ]:
!python src/validate_db.py

### 9. Collection statistics -- Clause vs BOQ

`validate_db.py` above confirms the collection is queryable but only shows one overall sample record; this breaks the total down by `chunk_type` and pulls one sample of each kind, confirming both the Clause and the BOQ pipeline actually wrote vectors.

In [ ]:
from src.storage import get_collection, COLLECTION_NAME

collection = get_collection()
total = collection.count()

# Chroma's .get() has no server-side group-by -- pull every metadata dict and
# count client-side; the corpus here is small enough (a few hundred chunks) that
# this is cheap.
all_meta = collection.get(include=["metadatas"])["metadatas"]
n_clause = sum(1 for m in all_meta if m.get("chunk_type") == "clause")
n_boq = sum(1 for m in all_meta if m.get("chunk_type") == "boq")
n_other = total - n_clause - n_boq

print(f"Collection            : {COLLECTION_NAME}")
print(f"Collection metadata   : {collection.metadata}")
print(f"Embedding model       : {collection.metadata.get('embedding_model')}")
print(f"Total vectors         : {total}")
print(f"  Clause vectors      : {n_clause}")
print(f"  BOQ vectors         : {n_boq}")
print(f"  Other chunk_type    : {n_other}")

sample_clause = collection.get(where={"chunk_type": "clause"}, limit=1, include=["documents", "metadatas"])
sample_boq = collection.get(where={"chunk_type": "boq"}, limit=1, include=["documents", "metadatas"])

print("\nSample CLAUSE chunk:")
if sample_clause["ids"]:
    print(" id      :", sample_clause["ids"][0])
    print(" text    :", sample_clause["documents"][0][:200])
    print(" clause_no / heading:", sample_clause["metadatas"][0].get("clause_no"), "/",
          sample_clause["metadatas"][0].get("heading"))
else:
    print(" (none found)")

print("\nSample BOQ chunk:")
if sample_boq["ids"]:
    print(" id      :", sample_boq["ids"][0])
    print(" text    :", sample_boq["documents"][0][:200])
    print(" item_type / s_no:", sample_boq["metadatas"][0].get("item_type"), "/",
          sample_boq["metadatas"][0].get("s_no"))
else:
    print(" (none found)")

assert total > 0, "Collection is empty -- ingestion failed."
assert n_clause > 0, "No clause vectors stored -- clause ingestion failed."
assert n_boq > 0, "No BOQ vectors stored -- BOQ ingestion failed (this is the pipeline this notebook exists to verify)."
print("\n[OK] Clause and BOQ vectors are both present in the collection.")

### 10. Retrieval validation queries

Runs every query below through the full non-LLM pipeline (`hybrid_retriever.hybrid_search()` -> `reranker.rerank()`), the same two stages `app.py`'s `/ask` endpoint uses before handing off to Gemma in notebook 2. Confirms clause queries surface clause chunks, BOQ queries surface BOQ chunks, and hybrid queries surface both.

In [ ]:
import src.hybrid_retriever as hr
import src.reranker as rr

CLAUSE_QUERIES = [
    "What are the requirements for fire alarm systems?",
    "Explain Clause 6.7.2.",
    "What are the testing requirements?",
    "Explain signalling requirements.",
]

BOQ_QUERIES = [
    "Find BOQ Item 1.01.",
    "Find MCC panels.",
    "Find UPS items.",
    "Show Schedule A panels.",
    "Find quantity for BOQ Item 2.03.",
]

HYBRID_QUERIES = [
    "Explain Fire Alarm Control Panels and corresponding BOQ items.",
    "Which BOQ item corresponds to signalling equipment?",
    "Show specification and BOQ entries for MCC panels.",
]


def run_category(label, queries):
    print("=" * 74)
    print(label)
    print("=" * 74)
    any_hit = False
    for q in queries:
        hits = hr.hybrid_search(q)
        reranked = rr.rerank(q, hits)
        print(f"\nQuery: {q}")
        if not reranked:
            print("  No results.")
            continue
        any_hit = True
        top = reranked[0]
        metadata = top.get("metadata") or {}
        chunk_type = metadata.get("chunk_type", "?")
        print(f"  Top result: chunk_id={top.get('chunk_id')}  "
              f"chunk_type={chunk_type}  reranker_score={top.get('reranker_score')}")
        if chunk_type == "boq":
            print(f"    item_type={metadata.get('item_type')}  s_no={metadata.get('s_no')}  "
                  f"panel_reference={metadata.get('panel_reference')}")
        else:
            print(f"    clause_no={metadata.get('clause_no')}  heading={metadata.get('heading')}")
        preview = (top.get("document") or "")[:200]
        print(f"    text: {preview}")
    return any_hit


clause_ok = run_category("CLAUSE QUERIES", CLAUSE_QUERIES)
boq_ok = run_category("BOQ QUERIES", BOQ_QUERIES)
hybrid_ok = run_category("HYBRID QUERIES", HYBRID_QUERIES)

assert clause_ok, "No clause query returned any result."
assert boq_ok, "No BOQ query returned any result."
assert hybrid_ok, "No hybrid query returned any result."
print("\n[OK] Clause, BOQ, and hybrid retrieval all returned results.")

---
### All validations passed

The fresh `chroma_db/` contains both clause and BOQ vectors, and dense + BM25 + reranked retrieval all work end-to-end for clause, BOQ, and hybrid queries. Switch this Colab runtime to **GPU** and open **`02_Gemma_Inference_and_Serving.ipynb`** to load Gemma 2 9B and serve `/ask` over the FastAPI app -- that notebook loads this same `chroma_db/` as-is and never rebuilds it.